In [5]:

"""
=============================================================================
MODULE 1 — DATA PIPELINE
Capstone Project: Zepto Data & AI Platform
=============================================================================
This script performs the complete data-engineering pipeline:
  Task 1: Scrape books from books.toscrape.com (≥60 books, ≥3 categories)
  Task 2: Clean and type-convert all fields
  Task 3: Currency conversion (GBP → INR at fixed rate 1 GBP = 105.50 INR)
  Task 4: Design & create normalized SQLite schema (2 tables with PK/FK)
  Task 5: Execute 5+ SQL queries (SELECT/WHERE, ORDER BY, LIMIT, DISTINCT, IN/BETWEEN, JOIN)
  Task 6: Verify JOIN using pd.read_sql vs pd.merge
=============================================================================
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import os



# Set the base directory for your project
BASE_DIR = r"C:\Users\mathepm\Desktop\Paul Alex AIML Capstone Project"
DATA_DIR = os.path.join(BASE_DIR, "data_pipeline")
# Create the folder if it doesn't exist (safety check)
os.makedirs(DATA_DIR, exist_ok=True)
# ============================================================================
# TASK 1: WEB SCRAPING
# ============================================================================

print("=" * 70)
print("TASK 1: WEB SCRAPING FROM books.toscrape.com")
print("=" * 70)

BASE_URL = "https://books.toscrape.com"


def get_soup(url):
    """Send GET request and return BeautifulSoup object."""
    response = requests.get(url)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def get_category_links(base_url, num_categories=5):
    """
    Extract category names and URLs from the sidebar navigation.
    Returns list of tuples: (category_name, category_url)
    """
    soup = get_soup(base_url)
    sidebar = soup.find("ul", class_="nav-list")
    category_items = sidebar.find("ul").find_all("li")

    categories = []
    for item in category_items[:num_categories]:
        link = item.find("a")
        name = link.text.strip()
        url = base_url + "/" + link["href"]
        categories.append((name, url))

    return categories


def scrape_category(category_name, category_url):
    """
    Scrape all books from a category, handling pagination.
    Returns list of dictionaries with book details.
    """
    books = []
    page_url = category_url

    while page_url:
        soup = get_soup(page_url)
        articles = soup.find_all("article", class_="product_pod")

        for article in articles:
            # Title
            title = article.find("h3").find("a")["title"]

            # Price (as listed, in GBP — includes £ symbol)
            price = article.find("p", class_="price_color").text.strip()

            # Star Rating (text form: One, Two, Three, Four, Five)
            rating_tag = article.find("p", class_="star-rating")
            star_rating = rating_tag["class"][1]  # Second class = rating word

            # Availability (as listed text)
            avail_tag = article.find("p", class_="instock availability")
            availability = avail_tag.text.strip() if avail_tag else "Not available"

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        # Check for next page
        next_btn = soup.find("li", class_="next")
        if next_btn:
            next_page = next_btn.find("a")["href"]
            page_url = "/".join(page_url.split("/")[:-1]) + "/" + next_page
        else:
            page_url = None

        time.sleep(0.5)  # Polite delay between requests

    return books


# --- Execute Scraping ---
print("\nFetching category links...")
categories = get_category_links(BASE_URL, num_categories=5)

print(f"Categories to scrape ({len(categories)}):")
for name, url in categories:
    print(f"  • {name}")

all_books = []
for cat_name, cat_url in categories:
    print(f"\n  Scraping '{cat_name}'...", end=" ")
    books = scrape_category(cat_name, cat_url)
    print(f"→ {len(books)} books found")
    all_books.extend(books)

# Create raw DataFrame
df_raw = pd.DataFrame(all_books)

print(f"\n{'─' * 70}")
print(f"SCRAPING COMPLETE")
print(f"  Total books scraped: {len(df_raw)}")
print(f"  Categories: {df_raw['category'].nunique()}")
print(f"  Columns: {list(df_raw.columns)}")
print(f"{'─' * 70}")

print("\n--- Raw Data Sample (first 10 rows) ---")
print(df_raw.head(10).to_string(index=False))

# Save raw data
df_raw.to_csv(os.path.join(DATA_DIR, "scraped_books_raw.csv"), index=False)
print(f"\n✅ Raw data saved to: {os.path.join(DATA_DIR, 'scraped_books_raw.csv')}")


# ============================================================================
# TASK 2: DATA CLEANING
# ============================================================================

print("\n\n" + "=" * 70)
print("TASK 2: DATA CLEANING & TYPE CONVERSION")
print("=" * 70)

df = df_raw.copy()

# --- 2a. Strip currency symbol from price, convert to float ---
print("\n[2a] Converting price to float (price_gbp)...")


def parse_price(price_str):
    """Remove currency symbol (£ or Â£) and convert to float."""
    try:
        # Remove any non-numeric characters except '.'
        cleaned = price_str.replace("£", "").replace("Â", "").strip()
        return float(cleaned)
    except (ValueError, AttributeError):
        return None  # Will be handled by median imputation


df["price_gbp"] = df["price"].apply(parse_price)

# --- 2b. Convert text star rating to integer (1-5) ---
print("[2b] Converting star_rating to integer (rating)...")

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}


def parse_rating(rating_str):
    """Convert text rating to integer 1-5."""
    try:
        return RATING_MAP[rating_str]
    except (KeyError, TypeError):
        return None  # Will be handled by median imputation


df["rating"] = df["star_rating"].apply(parse_rating)

# --- 2c. Parse availability to boolean ---
print("[2c] Converting availability to boolean (in_stock)...")


def parse_availability(avail_str):
    """Convert availability text to boolean."""
    if isinstance(avail_str, str):
        return "in stock" in avail_str.lower()
    return False


df["in_stock"] = df["availability"].apply(parse_availability)

# --- 2d. Handle missing/failed values ---
print("[2d] Handling missing values...")

# Count nulls before imputation
null_price = df["price_gbp"].isnull().sum()
null_rating = df["rating"].isnull().sum()

print(f"     Null price_gbp values: {null_price}")
print(f"     Null rating values: {null_rating}")

# DECISION: Median imputation for numeric fields (price_gbp, rating)
# JUSTIFICATION: Median is robust to outliers. Dropping rows would reduce
# our dataset size, and since the task requires ≥60 rows, imputation is safer.
# For boolean in_stock, any parse failure defaults to False (conservative).

if null_price > 0:
    median_price = df["price_gbp"].median()
    df["price_gbp"].fillna(median_price, inplace=True)
    print(f"     → Imputed {null_price} price values with median: £{median_price:.2f}")

if null_rating > 0:
    median_rating = int(df["rating"].median())
    df["rating"].fillna(median_rating, inplace=True)
    print(f"     → Imputed {null_rating} rating values with median: {median_rating}")

# Ensure correct types
df["rating"] = df["rating"].astype(int)
df["in_stock"] = df["in_stock"].astype(bool)

print("\n--- Cleaned Data Types ---")
print(f"  price_gbp: {df['price_gbp'].dtype}")
print(f"  rating:    {df['rating'].dtype}")
print(f"  in_stock:  {df['in_stock'].dtype}")

print("\n--- Cleaned Data Sample ---")
print(df[["title", "price_gbp", "rating", "in_stock", "category"]].head(10).to_string(index=False))


# ============================================================================
# TASK 3: CURRENCY CONVERSION (GBP → INR)
# ============================================================================

print("\n\n" + "=" * 70)
print("TASK 3: CURRENCY CONVERSION (GBP → INR)")
print("=" * 70)

# Fixed baseline conversion rate (project-defined constant)
GBP_TO_INR = 105.50
print(f"\nFixed conversion rate: 1 GBP = {GBP_TO_INR} INR")
print("(Project-defined constant — not a live/historical market rate)")

df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)

print("\n--- Price Conversion Sample ---")
print(df[["title", "price_gbp", "price_inr"]].head(10).to_string(index=False))

print(f"\n  Min price: £{df['price_gbp'].min():.2f} = ₹{df['price_inr'].min():.2f}")
print(f"  Max price: £{df['price_gbp'].max():.2f} = ₹{df['price_inr'].max():.2f}")
print(f"  Avg price: £{df['price_gbp'].mean():.2f} = ₹{df['price_inr'].mean():.2f}")


# ============================================================================
# TASK 4: SQLITE DATABASE — NORMALIZED SCHEMA
# ============================================================================

print("\n\n" + "=" * 70)
print("TASK 4: SQLite DATABASE CREATION (Normalized Schema)")
print("=" * 70)

DB_PATH = os.path.join(DATA_DIR, "books_pipeline.db")

# Remove old database if exists (for clean re-runs)
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# --- Create Tables ---
print("\n[4a] Creating normalized schema...")

# Table 1: categories (parent table)
cursor.execute("""
    CREATE TABLE categories (
        category_id INTEGER PRIMARY KEY AUTOINCREMENT,
        category_name TEXT UNIQUE NOT NULL
    );
""")

# Table 2: books (child table with FK to categories)
cursor.execute("""
    CREATE TABLE books (
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        price_gbp REAL NOT NULL,
        price_inr REAL NOT NULL,
        rating INTEGER NOT NULL CHECK(rating BETWEEN 1 AND 5),
        in_stock INTEGER NOT NULL,
        category_id INTEGER NOT NULL,
        FOREIGN KEY (category_id) REFERENCES categories(category_id)
    );
""")

print("  ✅ Table 'categories' created (category_id PK, category_name UNIQUE)")
print("  ✅ Table 'books' created (book_id PK, category_id FK → categories)")

# --- Insert Data ---
print("\n[4b] Inserting data into tables...")

# Insert unique categories first
unique_categories = df["category"].unique()
for cat_name in unique_categories:
    cursor.execute("INSERT INTO categories (category_name) VALUES (?);", (cat_name,))

conn.commit()

# Build a mapping: category_name → category_id
cursor.execute("SELECT category_id, category_name FROM categories;")
cat_map = {name: cid for cid, name in cursor.fetchall()}

print(f"  Inserted {len(cat_map)} categories:")
for name, cid in cat_map.items():
    print(f"    {cid}: {name}")

# Insert books with category_id foreign key
books_inserted = 0
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?);
    """, (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        row["rating"],
        int(row["in_stock"]),  # SQLite stores bool as 0/1
        cat_map[row["category"]]
    ))
    books_inserted += 1

conn.commit()
print(f"\n  ✅ Inserted {books_inserted} books into the database")
print(f"  Database saved at: {DB_PATH}")


# ============================================================================
# TASK 5: SQL QUERIES (5+ queries covering required clauses)
# ============================================================================

print("\n\n" + "=" * 70)
print("TASK 5: SQL QUERIES")
print("=" * 70)


def run_query(conn, query_num, description, sql):
    """Execute a SQL query and display results."""
    print(f"\n{'─' * 70}")
    print(f"QUERY {query_num}: {description}")
    print(f"{'─' * 70}")
    print(f"SQL:\n{sql.strip()}")
    print(f"\nRESULT:")
    result = pd.read_sql(sql, conn)
    print(result.to_string(index=False))
    print(f"({len(result)} rows returned)")
    return result


# --- QUERY 1: SELECT, WHERE, ORDER BY, LIMIT ---
# "Top 5 most expensive books that are in stock"
sql_1 = """
    SELECT title, price_gbp, price_inr, rating
    FROM books
    WHERE in_stock = 1
    ORDER BY price_gbp DESC
    LIMIT 5;
"""
q1 = run_query(conn, 1, "Top 5 most expensive books in stock (SELECT, WHERE, ORDER BY, LIMIT)", sql_1)


# --- QUERY 2: DISTINCT ---
# "All distinct ratings available in the catalogue"
sql_2 = """
    SELECT DISTINCT rating
    FROM books
    ORDER BY rating;
"""
q2 = run_query(conn, 2, "All distinct ratings in the catalogue (DISTINCT)", sql_2)


# --- QUERY 3: IN ---
# "Books with rating of 4 or 5 (highly rated)"
sql_3 = """
    SELECT title, rating, price_gbp, price_inr
    FROM books
    WHERE rating IN (4, 5)
    ORDER BY rating DESC, price_gbp DESC
    LIMIT 10;
"""
q3 = run_query(conn, 3, "Highly rated books (rating 4 or 5) — uses IN", sql_3)


# --- QUERY 4: BETWEEN ---
# "Books priced between £20 and £40"
sql_4 = """
    SELECT title, price_gbp, price_inr, rating
    FROM books
    WHERE price_gbp BETWEEN 20 AND 40
    ORDER BY price_gbp ASC
    LIMIT 10;
"""
q4 = run_query(conn, 4, "Books priced between £20 and £40 (BETWEEN)", sql_4)


# --- QUERY 5: JOIN ---
# "Top 10 highest-priced books per category (JOIN between books & categories)"
sql_5 = """
    SELECT
        c.category_name,
        b.title,
        b.price_gbp,
        b.price_inr,
        b.rating,
        b.in_stock
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    ORDER BY c.category_name, b.price_gbp DESC
    LIMIT 10;
"""
q5 = run_query(conn, 5, "Top books per category — JOIN between books & categories", sql_5)


# --- QUERY 6 (BONUS): Aggregation with JOIN ---
# "Average price and count per category"
sql_6 = """
    SELECT
        c.category_name,
        COUNT(b.book_id) AS total_books,
        ROUND(AVG(b.price_gbp), 2) AS avg_price_gbp,
        ROUND(AVG(b.price_inr), 2) AS avg_price_inr,
        ROUND(AVG(b.rating), 1) AS avg_rating
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    GROUP BY c.category_name
    ORDER BY avg_price_gbp DESC;
"""
q6 = run_query(conn, 6, "Average price & rating per category (JOIN + GROUP BY)", sql_6)


# ============================================================================
# TASK 6: pd.read_sql vs pd.merge COMPARISON
# ============================================================================

print("\n\n" + "=" * 70)
print("TASK 6: pd.read_sql vs pd.merge COMPARISON")
print("=" * 70)

# --- Approach 1: Using pd.read_sql (SQL JOIN) ---
print("\n[6a] JOIN result via pd.read_sql (SQL handles the join):")
print("─" * 70)

sql_join = """
    SELECT
        b.book_id,
        b.title,
        b.price_gbp,
        b.price_inr,
        b.rating,
        b.in_stock,
        c.category_name
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    ORDER BY b.book_id
    LIMIT 15;
"""
df_sql_join = pd.read_sql(sql_join, conn)
print(df_sql_join.to_string(index=False))
print(f"Shape: {df_sql_join.shape}")

# --- Approach 2: Using pd.merge (Python handles the join) ---
print("\n\n[6b] JOIN result via pd.merge (pandas handles the join):")
print("─" * 70)

# Read both tables into DataFrames separately
df_books = pd.read_sql("SELECT * FROM books ORDER BY book_id LIMIT 15;", conn)
df_categories = pd.read_sql("SELECT * FROM categories;", conn)

# Perform the merge (equivalent to SQL JOIN on category_id)
df_merged = pd.merge(
    df_books,
    df_categories,
    on="category_id",
    how="inner"
)

# Select same columns in same order for comparison
df_merged_display = df_merged[["book_id", "title", "price_gbp", "price_inr",
                                "rating", "in_stock", "category_name"]]

print(df_merged_display.to_string(index=False))
print(f"Shape: {df_merged_display.shape}")
# --- Side-by-Side Comparison Display ---
print("\n\n[6b+] SIDE-BY-SIDE COMPARISON (pd.read_sql vs pd.merge):")
print("─" * 70)

# Combine both DataFrames side by side with labeled column headers
df_side_by_side = pd.concat(
    [df_sql_join.reset_index(drop=True), df_merged_display.reset_index(drop=True)],
    axis=1,
    keys=["pd.read_sql (SQL JOIN)", "pd.merge (Python JOIN)"]
)

print(df_side_by_side.to_string(index=False))
print(f"\n  Left block:  pd.read_sql result ({df_sql_join.shape[0]} rows × {df_sql_join.shape[1]} cols)")
print(f"  Right block: pd.merge result ({df_merged_display.shape[0]} rows × {df_merged_display.shape[1]} cols)")
print(f"  ✅ Both outputs are displayed side by side for visual comparison.")


# --- Verify both approaches produce equivalent output ---
print("\n\n[6c] VERIFICATION — Are both outputs equivalent?")
print("─" * 70)

# Reset indices for comparison
df_sql_compare = df_sql_join.reset_index(drop=True)
df_merge_compare = df_merged_display.reset_index(drop=True)

# Check equality
are_equal = df_sql_compare.equals(df_merge_compare)
print(f"\n  pd.read_sql shape:  {df_sql_compare.shape}")
print(f"  pd.merge shape:     {df_merge_compare.shape}")
print(f"  Columns match:      {list(df_sql_compare.columns) == list(df_merge_compare.columns)}")
print(f"  Values match:       {are_equal}")

if are_equal:
    print("\n  ✅ VERIFIED: Both approaches produce IDENTICAL results!")
else:
    # Show differences if any (unlikely but good to check)
    print("\n  ⚠️  Minor differences found. Checking column by column:")
    for col in df_sql_compare.columns:
        match = (df_sql_compare[col] == df_merge_compare[col]).all()
        print(f"    {col}: {'✅ Match' if match else '❌ Mismatch'}")


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n\n" + "=" * 70)
print("MODULE 1 — PIPELINE COMPLETE ✅")
print("=" * 70)
print(f"""
Summary:
  • Books scraped:      {len(df)} (from {df['category'].nunique()} categories)
  • Fields created:     price_gbp (float), rating (int 1-5), in_stock (bool), price_inr (float)
  • Conversion rate:    1 GBP = {GBP_TO_INR} INR (fixed project-defined constant)
  • Database:           {DB_PATH}
  • Schema:             2 tables (categories → books) with PK/FK relationship
  • SQL queries:        6 queries covering SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, IN, BETWEEN, JOIN
  • Verification:       pd.read_sql and pd.merge produce identical JOIN results
""")

# Close the database connection
conn.close()
print("Database connection closed.")



TASK 1: WEB SCRAPING FROM books.toscrape.com

Fetching category links...
Categories to scrape (5):
  • Travel
  • Mystery
  • Historical Fiction
  • Sequential Art
  • Classics

  Scraping 'Travel'... → 11 books found

  Scraping 'Mystery'... → 32 books found

  Scraping 'Historical Fiction'... → 26 books found

  Scraping 'Sequential Art'... → 75 books found

  Scraping 'Classics'... → 19 books found

──────────────────────────────────────────────────────────────────────
SCRAPING COMPLETE
  Total books scraped: 163
  Categories: 5
  Columns: ['title', 'price', 'star_rating', 'availability', 'category']
──────────────────────────────────────────────────────────────────────

--- Raw Data Sample (first 10 rows) ---
                                                                                            title   price star_rating availability category
                                                                          It's Only the Himalayas Â£45.17         Two     In stock   Trav